# Men's Starters

Get ratings of each team's starters in a given year. This file needs to be ran for every relevant season. 

### Data Setup

In [1]:
import pandas as pd

season = 2025

df = pd.read_parquet(f'../data/unprocessed/mens_starters/starters_{season}.parquet')

df

,Date,Team,Location,Opponent,Result,Overtime,Team Score,Opponent Score,Team Starters,Opponent Starters,...,Team Starter 3,Team Starter 4,Team Starter 5,Opponent Starter 1,Opponent Starter 2,Opponent Starter 3,Opponent Starter 4,Opponent Starter 5,Score Differential,Adjusted Score Differential
0,2024-11-09,Abilene Christian,1,Middle Tennessee,-1,0,56,79,L. Bettiol · N. DeGruy · H. Madden · J. Venzan...,J. Bufford · K. Lands · E. Mostafa · J. Porter...,...,Abilene Christian H. Madden,Abilene Christian J. Venzant,Abilene Christian Q. Williams,Middle Tennessee J. Bufford,Middle Tennessee K. Lands,Middle Tennessee E. Mostafa,Middle Tennessee J. Porter,Middle Tennessee C. Weston,-23,1.500000
1,2024-11-16,Abilene Christian,1,Texas State,1,0,72,60,N. DeGruy · C. Hornecker · H. Madden · J. Venz...,D. Drinnon · K. Gumbs · J. O'Garro · T. Pope ·...,...,Abilene Christian H. Madden,Abilene Christian J. Venzant,Abilene Christian Q. Williams,Texas State D. Drinnon,Texas State K. Gumbs,Texas State J. O'Garro,Texas State T. Pope,Texas State C. Turner,12,1.417062
2,2024-11-20,Abilene Christian,-1,Kennesaw State,-1,0,78,84,L. Bettiol · N. DeGruy · H. Madden · J. Venzan...,S. Cottle · B. Lue · J. Miller · A. Weir · A. ...,...,Abilene Christian H. Madden,Abilene Christian J. Venzant,Abilene Christian Q. Williams,Kennesaw State S. Cottle,Kennesaw State B. Lue,Kennesaw State J. Miller,Kennesaw State A. Weir,Kennesaw State A. Wooley,-6,1.285760
3,2024-11-25,Abilene Christian,0,Southern Mississippi,1,0,82,74,L. Bettiol · N. DeGruy · H. Madden · J. Venzan...,N. Alvarez · D. Harris · C. Montgomery · C. Wa...,...,Abilene Christian H. Madden,Abilene Christian J. Venzant,Abilene Christian Q. Williams,Southern Mississippi N. Alvarez,Southern Mississippi D. Harris,Southern Mississippi C. Montgomery,Southern Mississippi C. Watson,Southern Mississippi L. Yat,8,1.338710
4,2024-11-26,Abilene Christian,-1,Montana State,-1,0,59,85,L. Bettiol · N. DeGruy · B. Hubbard · H. Madde...,M. Agbonkpolo · P. McMahon · J. Mullins · T. P...,...,Abilene Christian B. Hubbard,Abilene Christian H. Madden,Abilene Christian Q. Williams,Montana State M. Agbonkpolo,Montana State P. McMahon,Montana State J. Mullins,Montana State T. Patterson,Montana State B. Walker,-26,1.500000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11277,2025-02-23,Youngstown State,1,Green Bay,1,0,81,77,C. Carroll · N. Galette · T. Harper · J. Maxey...,M. Hall · J. Johnson · Y. Levy · P. Ruedinger ...,...,Youngstown State T. Harper,Youngstown State J. Maxey,Youngstown State S. Uijtendaal,Green Bay M. Hall,Green Bay J. Johnson,Green Bay Y. Levy,Green Bay P. Ruedinger,Green Bay B. Tweedy,4,1.214668
11278,2025-03-01,Youngstown State,-1,Northern Kentucky,-1,0,79,88,C. Carroll · N. Galette · T. Harper · J. Maxey...,J. Dilling · D. Gherezgher · K. Itejere · T. R...,...,Youngstown State T. Harper,Youngstown State J. Maxey,Youngstown State S. Uijtendaal,Northern Kentucky J. Dilling,Northern Kentucky D. Gherezgher,Northern Kentucky K. Itejere,Northern Kentucky T. Robinson,Northern Kentucky S. Vinson,-9,1.361013
11279,2025-03-06,Youngstown State,1,Purdue Fort Wayne,1,0,72,67,C. Carroll · N. Galette · T. Harper · J. Maxey...,R. Bello · J. Jackson · Q. Morton-Robertson · ...,...,Youngstown State T. Harper,Youngstown State J. Maxey,Youngstown State S. Uijtendaal,Purdue Fort Wayne R. Bello,Purdue Fort Wayne J. Jackson,Purdue Fort Wayne Q. Morton-Robertson,Purdue Fort Wayne E. Mulder,Purdue Fort Wayne M. Nelson,5,1.253292
11280,2025-03-10,Youngstown State,0,Cleveland State,1,0,56,54,C. Carroll · N. Galette · T. Harper · J. Maxey...,I. Abidde · D. Arnett · E. Dibba · T. Smith · ...,...,Youngstown State T. Harper,Youngstown State J. Maxey,Youngstown State S. Uijtendaal,Cleveland State I. Abidde,Cleveland State D. Arnett,Cleveland State E. Dibba,Cleveland State T. Smith,Cleveland State T. Staveskie,2,1.102120


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11282 entries, 0 to 11281
Data columns (total 22 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   Date                         11282 non-null  datetime64[ns]
 1   Team                         11282 non-null  object        
 2   Location                     11282 non-null  int8          
 3   Opponent                     11282 non-null  object        
 4   Result                       11282 non-null  int8          
 5   Overtime                     11282 non-null  int8          
 6   Team Score                   11282 non-null  int64         
 7   Opponent Score               11282 non-null  int64         
 8   Team Starters                11282 non-null  object        
 9   Opponent Starters            11282 non-null  object        
 10  Team Starter 1               11282 non-null  object        
 11  Team Starter 2               11282 non-nu

### Model Building

##### Starters Ratings

Unfortunately, there is no way to differentiate between 2 starters with the same name on the same team. In such circumstances, we double count the entity. 

In [3]:
X = (
    pd.get_dummies(df['Team Starter 1']).astype('int8')
).add(
    pd.get_dummies(df['Team Starter 2']).astype('int8'), fill_value=0
).add(
    pd.get_dummies(df['Team Starter 3']).astype('int8'), fill_value=0 
).add(
    pd.get_dummies(df['Team Starter 4']).astype('int8'), fill_value=0 
).add(
    pd.get_dummies(df['Team Starter 5']).astype('int8'), fill_value=0 
).add(
    -pd.get_dummies(df['Opponent Starter 1']).astype('int8'), fill_value=0 
).add(
    -pd.get_dummies(df['Opponent Starter 2']).astype('int8'), fill_value=0 
).add(
    -pd.get_dummies(df['Opponent Starter 3']).astype('int8'), fill_value=0 
).add(
    -pd.get_dummies(df['Opponent Starter 4']).astype('int8'), fill_value=0 
).add(
    -pd.get_dummies(df['Opponent Starter 5']).astype('int8'), fill_value=0 
)

X['Home Field Advantage'] = df['Location'].copy()

X

C:\Users\mhugh\AppData\Local\Temp\ipykernel_16316\563957636.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X['Home Field Advantage'] = df['Location'].copy()


,Abilene Christian B. Hubbard,Abilene Christian C. Hornecker,Abilene Christian H. Madden,Abilene Christian J. Venzant,Abilene Christian L. Bettiol,Abilene Christian M. Hill,Abilene Christian N. DeGruy,Abilene Christian Q. Williams,Abilene Christian R. Smith,Abilene Christian Y. Rivera,...,Yale Y. Gharram,Youngstown State C. Carroll,Youngstown State E. Farmer,Youngstown State G. Dynes,Youngstown State J. Maxey,Youngstown State J. Nelson,Youngstown State N. Galette,Youngstown State S. Uijtendaal,Youngstown State T. Harper,Home Field Advantage
0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
1,0.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1
3,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11277,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,1
11278,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,-1
11279,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,1
11280,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,0


In [4]:
import numpy as np
from sklearn.model_selection import GroupKFold

def get_gkf_data(X, y, w, groups, cv=3):
    """
    Converts training data to list of folds
    """
    np.random.seed(22)
    gkf = GroupKFold(n_splits=cv)

    data = []
    for train_index, test_index in gkf.split(X, y, groups=groups):
        X_train = X[train_index]
        X_test = X[test_index]

        y_train = y[train_index]
        y_test = y[test_index]

        # sample weights
        w_train = w[train_index]

        data.append((X_train, X_test, y_train, y_test, w_train))

    return data

cv_data = get_gkf_data(X.to_numpy(), df['Result'].to_numpy(), df['Adjusted Score Differential'].to_numpy(), df['Date'].to_numpy())

len(cv_data)

3

In [5]:
def rescale_weights(arr, minimum, maximum):
    """
    Rescale the weights array to match desired weights.
    Assumes the data is already transformed where a 1 point win is 1.00, a blowout is 1.50, and a tie is 0.50.
    Ties will be weighted as half the minimum.
    """

    arr_scaled = ((arr - 1.00) / 0.50) * (maximum - minimum) + minimum
    arr_scaled[arr == 0.50] = minimum / 2
    return arr_scaled

In [6]:
from sklearn.metrics import log_loss
from sklearn.linear_model import LogisticRegression
import warnings
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial, cv_data=cv_data):
    # model tuning
    C = trial.suggest_float('C', 0.1, 10, log=True)
    mod = LogisticRegression(penalty='l2', C=C, fit_intercept=False)
    minimum = trial.suggest_float('minimum', 0.1, 1.0, step=0.1)
    maximum = trial.suggest_float('maximum', 1.0, 8.0, step=0.5)

    # cross validation
    y_actuals = []
    y_preds = []
    for X_train, X_test, y_train, y_test, w_train in cv_data:
        y_actuals.append(y_test)

        weights = rescale_weights(w_train, minimum, maximum)

        with warnings.catch_warnings():
            warnings.filterwarnings('ignore')  # prevent convergence warnings
            mod.fit(X_train, y_train, sample_weight=weights)

        y_preds.append(mod.predict_proba(X_test)[:, 1])

    return log_loss(np.hstack(y_actuals), np.hstack(y_preds))

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=22))
study.optimize(objective, n_trials=100, show_progress_bar=True)

study.best_params

  0%|          | 0/100 [00:00<?, ?it/s]

{'C': 0.12506368595172224, 'minimum': 0.2, 'maximum': 1.5}

In [7]:
minimum = study.best_params['minimum']
maximum = study.best_params['maximum']

weight = rescale_weights(df['Adjusted Score Differential'], minimum, maximum)

mod = LogisticRegression(penalty='l2', C=study.best_params['C'], fit_intercept=False)

mod.fit(X, df['Result'], sample_weight=weight)

df_ratings = pd.DataFrame(
    {
        'Starter': X.columns,
        'Rating': mod.coef_[0]
    }
).sort_values(by=['Rating'], ascending=False, ignore_index=True)

df_ratings_display = df_ratings.loc[df_ratings['Starter'] != 'Home Field Advantage', :].reset_index(drop=True)
df_ratings_display.index += 1

df_ratings_display.head(25)

,Starter,Rating
1,Auburn D. Jones,0.827584
2,Houston J. Tugler,0.811120
3,Duke S. James,0.747559
4,Florida R. Chinyelu,0.710451
5,Florida W. Richard,0.710451
6,Florida W. Clayton,0.691742
7,Florida A. Condon,0.690261
8,Auburn D. Cardwell,0.687718
9,Duke K. Maluach,0.672309
10,Duke K. Knueppel,0.672309


### Miscellaneous

##### Save Rankings

Get last available lineup for each team

In [8]:
starter_to_rating = dict(zip(df_ratings_display['Starter'], df_ratings_display['Rating']))

len(starter_to_rating)

3281

In [9]:
df_last_starters = (
    df
    .sort_values(by=['Date']).groupby(['Team'])
    .tail(1)
    .reset_index(drop=True)[['Team'] + [f'Team Starter {i}' for i in range(1, 6)]]
)

df_last_starters

,Team,Team Starter 1,Team Starter 2,Team Starter 3,Team Starter 4,Team Starter 5
0,West Georgia,West Georgia K. Griffin,West Georgia M. Griffin,West Georgia M. Noel,West Georgia T. Releford,West Georgia S. Williams-Dryden
1,Bellarmine,Bellarmine T. Doyle,Bellarmine L. Hacker,Bellarmine C. Hopf,Bellarmine B. Smith,Bellarmine G. Whitaker
2,Western Illinois,Western Illinois T. Deveaux,Western Illinois M. Diouf,Western Illinois R. Myers,Western Illinois J. Rollins,Western Illinois S. Smith
3,Mercyhurst,Mercyhurst B. Blunt,Mercyhurst M. Jusianiec,Mercyhurst J. Planutis,Mercyhurst S. Rathan-Mayes,Mercyhurst A. Reichert
4,Southern Indiana,Southern Indiana J. Campion,Southern Indiana R. Hall,Southern Indiana J. Mielke,Southern Indiana S. Olowoniyi,Southern Indiana J. Randall
...,...,...,...,...,...,...
359,Wisconsin,Wisconsin J. Blackwell,Wisconsin S. Crowl,Wisconsin M. Klesmit,Wisconsin J. Tonje,Wisconsin N. Winter
360,Yale,Yale S. Aletan,Yale B. Mbeng,Yale J. Poulakidas,Yale C. Simmons,Yale N. Townsend
361,Florida,Florida R. Chinyelu,Florida W. Clayton,Florida A. Condon,Florida A. Martin,Florida W. Richard
362,Tennessee,Tennessee C. Lanier,Tennessee J. Mashack,Tennessee I. Milicic,Tennessee F. Okpara,Tennessee Z. Zeigler


In [10]:
for i in range(1, 6):
    df_last_starters[f'Team Starter {i} Rating'] = df_last_starters[f'Team Starter {i}'].map(starter_to_rating)

df_last_starters['Rating'] = df_last_starters[[f'Team Starter {i} Rating' for i in range(1, 6)]].mean(axis=1)

df_last_starters

,Team,Team Starter 1,Team Starter 2,Team Starter 3,Team Starter 4,Team Starter 5,Team Starter 1 Rating,Team Starter 2 Rating,Team Starter 3 Rating,Team Starter 4 Rating,Team Starter 5 Rating,Rating
0,West Georgia,West Georgia K. Griffin,West Georgia M. Griffin,West Georgia M. Noel,West Georgia T. Releford,West Georgia S. Williams-Dryden,-0.438690,-0.155234,-0.227835,-0.001667,-0.381402,-0.240966
1,Bellarmine,Bellarmine T. Doyle,Bellarmine L. Hacker,Bellarmine C. Hopf,Bellarmine B. Smith,Bellarmine G. Whitaker,-0.252545,0.216589,-0.407957,-0.538418,-0.194487,-0.235363
2,Western Illinois,Western Illinois T. Deveaux,Western Illinois M. Diouf,Western Illinois R. Myers,Western Illinois J. Rollins,Western Illinois S. Smith,0.030784,-0.109138,-0.375915,-0.257396,-0.338135,-0.209960
3,Mercyhurst,Mercyhurst B. Blunt,Mercyhurst M. Jusianiec,Mercyhurst J. Planutis,Mercyhurst S. Rathan-Mayes,Mercyhurst A. Reichert,-0.126179,-0.102943,-0.219585,-0.219585,-0.219585,-0.177575
4,Southern Indiana,Southern Indiana J. Campion,Southern Indiana R. Hall,Southern Indiana J. Mielke,Southern Indiana S. Olowoniyi,Southern Indiana J. Randall,-0.289822,0.078997,-0.292012,-0.238378,-0.189519,-0.186147
...,...,...,...,...,...,...,...,...,...,...,...,...
359,Wisconsin,Wisconsin J. Blackwell,Wisconsin S. Crowl,Wisconsin M. Klesmit,Wisconsin J. Tonje,Wisconsin N. Winter,0.400618,0.400618,0.533310,0.400618,0.400618,0.427156
360,Yale,Yale S. Aletan,Yale B. Mbeng,Yale J. Poulakidas,Yale C. Simmons,Yale N. Townsend,0.284555,0.312346,0.194228,0.503897,0.312346,0.321474
361,Florida,Florida R. Chinyelu,Florida W. Clayton,Florida A. Condon,Florida A. Martin,Florida W. Richard,0.710451,0.691742,0.690261,0.373558,0.710451,0.635293
362,Tennessee,Tennessee C. Lanier,Tennessee J. Mashack,Tennessee I. Milicic,Tennessee F. Okpara,Tennessee Z. Zeigler,0.621858,0.621858,0.417308,0.621858,0.417308,0.540038


In [11]:
df_sheet = df_last_starters[['Team', 'Rating']].copy().sort_values(by=['Rating'], ascending=False, ignore_index=True)

df_sheet.head(25)

,Team,Rating
0,Florida,0.635293
1,Auburn,0.611045
2,St. John's (NY),0.597470
3,Houston,0.580753
4,Duke,0.579505
5,Michigan State,0.544517
6,Tennessee,0.540038
7,Drake,0.500627
8,Virginia Commonwealth,0.480728
9,Saint Mary's (CA),0.461035


In [12]:
df_sheet.to_parquet(f'../data/preprocessed/mens_starters/starters_{season}.parquet')

'Done'

'Done'